# Ogrenci Dikkat Tespiti - Duygu Analizi Model Egitimi

## TUBITAK 2209-B Projesi

Bu notebook, ogrencilerin dikkat durumlarini tespit etmek icin **FER2013** veri seti uzerinde duygu analizi modeli egitir.

### Proje Ozeti
- **Veri Seti:** FER2013 (7 sinif -> 3 sinif: positive, neutral, negative)
- **Mimariler:** EfficientNet-B3, EfficientNet-B0, MobileNetV3-Large, ResNet50+CBAM
- **Egitim Stratejisi:** 2 fazli egitim (head-only + full fine-tuning)
- **Kayip Fonksiyonu:** Focal Loss (sinif dengesizligine karsi)
- **Cikti:** ONNX formatinda deploy-ready model

### Gereksinimler
- Google Colab (GPU runtime: T4 veya daha iyi)
- FER2013 veri seti (Kaggle'dan indirilecek)

In [ ]:
# ============================================================================
# Cell 1: Setup & Install
# ============================================================================
# Install required packages (PyTorch is pre-installed in Colab)

!pip install timm albumentations onnx onnxruntime pandas seaborn tqdm -q

# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ============================================================================
# Cell 2: Mount Google Drive
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

# Create project directory in Drive for saving results
import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/Tubitak-2209B'
os.makedirs(f'{DRIVE_PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT_DIR}/results', exist_ok=True)
print(f"Drive project directory: {DRIVE_PROJECT_DIR}")

In [ ]:
# ============================================================================
# Cell 3: Project Setup
# ============================================================================
# Option 1: Clone from GitHub (uncomment and update URL)
# !git clone https://github.com/your-username/Tubitak-2209B.git /content/project
# import sys
# sys.path.insert(0, '/content/project')

# Option 2: Upload project as zip from Google Drive
# !cp /content/drive/MyDrive/Tubitak-2209B/project.zip /content/
# !unzip -q /content/project.zip -d /content/project

# For this notebook, all necessary code is defined inline below,
# so no external project files are needed.

import sys
import os
import warnings
warnings.filterwarnings('ignore')

print("Project setup complete. All code is self-contained in this notebook.")

In [ ]:
# ============================================================================
# Cell 4: Imports & Configuration Constants
# ============================================================================

import json
import time
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

# ── Emotion Mapping (7 -> 3 classes) ──────────────────────────────────────
# FER2013 original: 0=Angry, 1=Disgust, 2=Fear, 3=Happy, 4=Sad, 5=Surprise, 6=Neutral
ORIGINAL_EMOTIONS = {
    0: "angry", 1: "disgust", 2: "fear", 3: "happy",
    4: "sad", 5: "surprise", 6: "neutral",
}

CLASS_NAMES = ["negative", "neutral", "positive"]
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = 3

# FER2013 original label -> 3-class index (surprise=None -> excluded)
FER_TO_3CLASS_IDX = {
    0: CLASS_TO_IDX["negative"],   # angry -> negative
    1: CLASS_TO_IDX["negative"],   # disgust -> negative
    2: CLASS_TO_IDX["negative"],   # fear -> negative
    3: CLASS_TO_IDX["positive"],   # happy -> positive
    4: CLASS_TO_IDX["negative"],   # sad -> negative
    5: None,                        # surprise -> excluded
    6: CLASS_TO_IDX["neutral"],    # neutral -> neutral
}

# ── Data Pipeline ─────────────────────────────────────────────────────────
IMAGE_SIZE = 224
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE = 32
NUM_WORKERS = 2

# Phase 1: Head-only training
PHASE1_EPOCHS = 5
PHASE1_LR = 1e-3

# Phase 2: Full fine-tuning
PHASE2_EPOCHS = 25
PHASE2_LR = 1e-4
WEIGHT_DECAY = 1e-4

EARLY_STOPPING_PATIENCE = 7
FOCAL_LOSS_GAMMA = 2.0

# ── Model Configs ─────────────────────────────────────────────────────────
MODEL_CONFIGS = {
    "efficientnet_b3": {
        "timm_name": "efficientnet_b3",
        "feature_dim": 1536, "hidden_dim": 512,
        "dropout": 0.3, "dropout2": 0.2,
    },
    "efficientnet_b0": {
        "timm_name": "efficientnet_b0",
        "feature_dim": 1280, "hidden_dim": 256,
        "dropout": 0.3, "dropout2": 0.2,
    },
    "mobilenet_v3": {
        "timm_name": "mobilenetv3_large_100",
        "feature_dim": 1280, "hidden_dim": 256,
        "dropout": 0.3, "dropout2": 0.2,
    },
    "resnet50_cbam": {
        "timm_name": "resnet50",
        "feature_dim": 2048, "hidden_dim": 512,
        "dropout": 0.3, "dropout2": 0.2,
    },
}

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Configuration loaded: {NUM_CLASSES} classes = {CLASS_NAMES}")

In [ ]:
# ============================================================================
# Cell 5: Dataset & Transforms
# ============================================================================

import albumentations as A
from albumentations.pytorch import ToTensorV2


def get_train_transforms():
    """Augmentation pipeline for training data."""
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.CoarseDropout(
            max_holes=1,
            max_height=int(IMAGE_SIZE * 0.2),
            max_width=int(IMAGE_SIZE * 0.2),
            min_height=int(IMAGE_SIZE * 0.05),
            min_width=int(IMAGE_SIZE * 0.05),
            fill_value=0,
            p=0.3,
        ),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def get_val_transforms():
    """Augmentation pipeline for validation/test data."""
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


class FERDataset(Dataset):
    """Dataset for FER2013 with 7->3 class mapping.

    Supports both CSV pixel format and image folder format.
    Surprise samples are excluded.
    """

    def __init__(self, root=None, csv_path=None, transform=None, usage=None):
        self.transform = transform
        self.images = []
        self.labels = []
        self._from_csv = csv_path is not None

        if csv_path is not None:
            self._load_csv(csv_path, usage)
        elif root is not None:
            self._load_folder(root)
        else:
            raise ValueError("Either 'root' or 'csv_path' must be provided.")

    def _load_csv(self, csv_path, usage=None):
        """Load from FER2013 CSV with pixel strings."""
        df = pd.read_csv(csv_path)
        if usage is not None:
            df = df[df["Usage"] == usage]

        for _, row in df.iterrows():
            original_label = int(row["emotion"])
            mapped = FER_TO_3CLASS_IDX.get(original_label)
            if mapped is None:
                continue  # skip surprise

            pixels = np.fromstring(row["pixels"], dtype=np.uint8, sep=" ")
            img = pixels.reshape(48, 48)
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

            self.images.append(img)
            self.labels.append(mapped)

    def _load_folder(self, root):
        """Load from image folder structure (negative/neutral/positive dirs)."""
        root = Path(root)
        class_map = {"negative": 0, "neutral": 1, "positive": 2}
        for class_name, label in class_map.items():
            class_dir = root / class_name
            if not class_dir.exists():
                continue
            for img_path in sorted(class_dir.iterdir()):
                if img_path.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}:
                    self.images.append(img_path)
                    self.labels.append(label)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        label = self.labels[idx]

        if self._from_csv:
            image = self.images[idx]  # already numpy RGB
        else:
            path = self.images[idx]
            image = cv2.imread(str(path))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label


def get_class_weights(dataset):
    """Compute class weights inversely proportional to frequency."""
    label_counts = Counter(dataset.labels)
    total = len(dataset.labels)
    num_classes = max(label_counts.keys()) + 1
    weights = torch.zeros(num_classes, dtype=torch.float32)
    for cls_idx in range(num_classes):
        count = label_counts.get(cls_idx, 1)
        weights[cls_idx] = total / (num_classes * count)
    return weights


def create_weighted_sampler(dataset):
    """Create WeightedRandomSampler for balanced sampling."""
    label_counts = Counter(dataset.labels)
    total = len(dataset.labels)
    class_weights = {cls: total / count for cls, count in label_counts.items()}
    sample_weights = [class_weights[label] for label in dataset.labels]
    return WeightedRandomSampler(weights=sample_weights, num_samples=total, replacement=True)


print("Dataset and transforms defined.")

In [ ]:
# ============================================================================
# Cell 6: Download / Load FER2013 Data
# ============================================================================
# Choose ONE of the options below:

# ── Option A: Kaggle API (recommended) ────────────────────────────────────
# 1. Go to kaggle.com -> Account -> Create New API Token
# 2. Upload the kaggle.json file when prompted

# from google.colab import files
# files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d msambare/fer2013 -p /content/data
# !unzip -q /content/data/fer2013.zip -d /content/data

# ── Option B: Manual upload ───────────────────────────────────────────────
# 1. Download fer2013.csv from Kaggle manually
# 2. Upload it using the file upload widget

# from google.colab import files
# uploaded = files.upload()  # Upload fer2013.csv
# !mkdir -p /content/data
# !mv fer2013.csv /content/data/

# ── Option C: From Google Drive ───────────────────────────────────────────
# If you already have the CSV in your Drive:
# !cp /content/drive/MyDrive/datasets/fer2013.csv /content/data/

# ── Set the path to your CSV file ─────────────────────────────────────────
CSV_PATH = "/content/data/fer2013.csv"

# Verify the file exists
if os.path.exists(CSV_PATH):
    df_check = pd.read_csv(CSV_PATH, nrows=5)
    print(f"FER2013 CSV loaded successfully!")
    print(f"Columns: {list(df_check.columns)}")
    print(f"Sample row emotion values: {df_check['emotion'].tolist()}")
else:
    print(f"WARNING: {CSV_PATH} not found!")
    print("Please use one of the options above to download/upload the FER2013 dataset.")

In [ ]:
# ============================================================================
# Cell 7: Prepare Data - 7->3 Class Mapping & Stratified Split
# ============================================================================

def load_and_map_fer2013(csv_path):
    """Load FER2013 CSV and map 7 classes to 3, excluding surprise."""
    df = pd.read_csv(csv_path)
    images, labels = [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Loading FER2013"):
        original_label = int(row["emotion"])
        mapped = FER_TO_3CLASS_IDX.get(original_label)
        if mapped is None:
            continue  # exclude surprise

        pixels = np.fromstring(row["pixels"], dtype=np.uint8, sep=" ")
        img = pixels.reshape(48, 48)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        images.append(img)
        labels.append(mapped)

    return images, labels


# Load and map
all_images, all_labels = load_and_map_fer2013(CSV_PATH)
print(f"\nTotal samples (surprise excluded): {len(all_images)}")

# Show original distribution
orig_counts = Counter(all_labels)
print("\nOverall class distribution:")
for idx, name in enumerate(CLASS_NAMES):
    count = orig_counts.get(idx, 0)
    pct = 100.0 * count / len(all_labels)
    print(f"  {name}: {count} ({pct:.1f}%)")

# Stratified train / val / test split
val_test_ratio = VAL_RATIO + TEST_RATIO
train_imgs, valtest_imgs, train_labels, valtest_labels = train_test_split(
    all_images, all_labels,
    test_size=val_test_ratio, stratify=all_labels, random_state=RANDOM_SEED,
)

relative_test_ratio = TEST_RATIO / val_test_ratio
val_imgs, test_imgs, val_labels, test_labels = train_test_split(
    valtest_imgs, valtest_labels,
    test_size=relative_test_ratio, stratify=valtest_labels, random_state=RANDOM_SEED,
)

print(f"\nSplit sizes: Train={len(train_labels)}, Val={len(val_labels)}, Test={len(test_labels)}")

# ── Visualize class distribution per split ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for ax, (split_name, split_labels) in zip(axes, [
    ("Train", train_labels), ("Validation", val_labels), ("Test", test_labels)
]):
    counts = Counter(split_labels)
    counts_list = [counts.get(i, 0) for i in range(NUM_CLASSES)]
    bars = ax.bar(CLASS_NAMES, counts_list, color=colors)
    ax.set_title(f"{split_name} Set (n={len(split_labels)})")
    ax.set_ylabel("Sample Count")
    for bar, count in zip(bars, counts_list):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
                str(count), ha='center', va='bottom', fontweight='bold')

plt.suptitle("Class Distribution per Split", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Cell 8: Visualize Sample Images
# ============================================================================

fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for row_idx, class_idx in enumerate(range(NUM_CLASSES)):
    # Collect sample indices for this class
    class_indices = [i for i, lbl in enumerate(train_labels) if lbl == class_idx]
    np.random.seed(RANDOM_SEED)
    sample_indices = np.random.choice(class_indices, size=5, replace=False)

    for col_idx, sample_idx in enumerate(sample_indices):
        ax = axes[row_idx, col_idx]
        img = train_imgs[sample_idx]
        ax.imshow(img)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(CLASS_NAMES[class_idx], fontsize=14, fontweight='bold', rotation=0, labelpad=60)

plt.suptitle("Sample Images per Class (48x48 original size)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Image shape (original): {train_imgs[0].shape}")
print(f"Image shape (after transform): will be ({3}, {IMAGE_SIZE}, {IMAGE_SIZE})")

In [ ]:
# ============================================================================
# Cell 9: Model Architectures (CBAM + Classifier Models)
# ============================================================================

# ── CBAM: Convolutional Block Attention Module ────────────────────────────

class ChannelAttention(nn.Module):
    """Channel attention sub-module of CBAM."""
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        mid = max(in_channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, in_channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        avg_out = self.mlp(self.avg_pool(x).view(b, c))
        max_out = self.mlp(self.max_pool(x).view(b, c))
        scale = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        return x * scale


class SpatialAttention(nn.Module):
    """Spatial attention sub-module of CBAM."""
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out, _ = x.max(dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        scale = self.sigmoid(self.conv(combined))
        return x * scale


class CBAM(nn.Module):
    """Convolutional Block Attention Module (Woo et al., ECCV 2018)."""
    def __init__(self, in_channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x


# ── Shared classifier head ────────────────────────────────────────────────

def _build_head(feature_dim, hidden_dim, dropout, dropout2):
    return nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Dropout(dropout),
        nn.Linear(feature_dim, hidden_dim),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout2),
        nn.Linear(hidden_dim, NUM_CLASSES),
    )


# ── Model Classes ─────────────────────────────────────────────────────────

class EfficientNetB3Classifier(nn.Module):
    """EfficientNet-B3 backbone with custom classifier head."""
    def __init__(self, pretrained=True):
        super().__init__()
        cfg = MODEL_CONFIGS["efficientnet_b3"]
        self.backbone = timm.create_model(cfg["timm_name"], pretrained=pretrained, num_classes=0, global_pool="")
        self.head = _build_head(cfg["feature_dim"], cfg["hidden_dim"], cfg["dropout"], cfg["dropout2"])

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True


class EfficientNetB0Classifier(nn.Module):
    """EfficientNet-B0 backbone with custom classifier head."""
    def __init__(self, pretrained=True):
        super().__init__()
        cfg = MODEL_CONFIGS["efficientnet_b0"]
        self.backbone = timm.create_model(cfg["timm_name"], pretrained=pretrained, num_classes=0, global_pool="")
        self.head = _build_head(cfg["feature_dim"], cfg["hidden_dim"], cfg["dropout"], cfg["dropout2"])

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True


class MobileNetV3Classifier(nn.Module):
    """MobileNetV3-Large backbone with custom classifier head."""
    def __init__(self, pretrained=True):
        super().__init__()
        cfg = MODEL_CONFIGS["mobilenet_v3"]
        self.backbone = timm.create_model(cfg["timm_name"], pretrained=pretrained, num_classes=0, global_pool="")
        self.head = _build_head(cfg["feature_dim"], cfg["hidden_dim"], cfg["dropout"], cfg["dropout2"])

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True


class ResNet50CBAMClassifier(nn.Module):
    """ResNet50 backbone with CBAM after each residual stage."""
    def __init__(self, pretrained=True):
        super().__init__()
        cfg = MODEL_CONFIGS["resnet50_cbam"]
        self.backbone = timm.create_model(cfg["timm_name"], pretrained=pretrained, num_classes=0, global_pool="")
        self.cbam1 = CBAM(256)
        self.cbam2 = CBAM(512)
        self.cbam3 = CBAM(1024)
        self.cbam4 = CBAM(2048)
        self.head = _build_head(cfg["feature_dim"], cfg["hidden_dim"], cfg["dropout"], cfg["dropout2"])

    def _forward_backbone(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.act1(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x); x = self.cbam1(x)
        x = self.backbone.layer2(x); x = self.cbam2(x)
        x = self.backbone.layer3(x); x = self.cbam3(x)
        x = self.backbone.layer4(x); x = self.cbam4(x)
        return x

    def forward(self, x):
        return self.head(self._forward_backbone(x))

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True


# ── Factory ───────────────────────────────────────────────────────────────

MODEL_REGISTRY = {
    "efficientnet_b3": EfficientNetB3Classifier,
    "efficientnet_b0": EfficientNetB0Classifier,
    "mobilenet_v3": MobileNetV3Classifier,
    "resnet50_cbam": ResNet50CBAMClassifier,
}

def create_model(model_name, pretrained=True):
    """Create a model by name."""
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{model_name}'. Available: {list(MODEL_REGISTRY.keys())}")
    return MODEL_REGISTRY[model_name](pretrained=pretrained)


# ── Focal Loss ────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    """Focal Loss for class imbalance (Lin et al., ICCV 2017)."""
    def __init__(self, gamma=FOCAL_LOSS_GAMMA, alpha=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        if alpha is not None:
            self.register_buffer("alpha", alpha.float())
        else:
            self.alpha = None

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp()
        targets_one_hot = F.one_hot(targets, num_classes=logits.size(1)).float()
        pt = (probs * targets_one_hot).sum(dim=1)
        log_pt = (log_probs * targets_one_hot).sum(dim=1)
        focal_weight = (1.0 - pt) ** self.gamma
        loss = -focal_weight * log_pt
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            loss = alpha_t * loss
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


# Quick verification
print("Model architectures defined:")
for name in MODEL_REGISTRY:
    cfg = MODEL_CONFIGS[name]
    print(f"  - {name}: backbone={cfg['timm_name']}, features={cfg['feature_dim']}")
print(f"\nFocal Loss: gamma={FOCAL_LOSS_GAMMA}")

In [ ]:
# ============================================================================
# Cell 10: Training Loop Functions
# ============================================================================

class InMemoryDataset(Dataset):
    """Wraps in-memory image arrays and labels into a PyTorch Dataset."""
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        if self.transform is not None:
            augmented = self.transform(image=image)
            image = augmented["image"]
        return image, label


def run_epoch(model, loader, criterion, optimizer, device, is_train=True):
    """Run one epoch of training or validation."""
    model.train() if is_train else model.eval()
    running_loss, correct, total = 0.0, 0, 0

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    pbar = tqdm(loader, desc="Train" if is_train else "Val", leave=False)

    with ctx:
        for images, targets in pbar:
            images = images.to(device)
            targets = targets.to(device)

            logits = model(images)
            loss = criterion(logits, targets)

            if is_train and optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += images.size(0)

            pbar.set_postfix(loss=running_loss/max(total,1), acc=correct/max(total,1))

    avg_loss = running_loss / max(total, 1)
    accuracy = correct / max(total, 1)
    return avg_loss, accuracy


def plot_training_curves(history, model_name):
    """Plot loss and accuracy curves inline."""
    epochs_range = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, history["train_loss"], 'b-o', markersize=3, label="Train")
    axes[0].plot(epochs_range, history["val_loss"], 'r-o', markersize=3, label="Val")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
    axes[0].set_title(f"{model_name} - Loss"); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(epochs_range, history["train_acc"], 'b-o', markersize=3, label="Train")
    axes[1].plot(epochs_range, history["val_acc"], 'r-o', markersize=3, label="Val")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
    axes[1].set_title(f"{model_name} - Accuracy"); axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.show()


def train_model(
    model_name,
    train_images, train_labels_list,
    val_images, val_labels_list,
    phase1_epochs=PHASE1_EPOCHS,
    phase2_epochs=PHASE2_EPOCHS,
    phase1_lr=PHASE1_LR,
    phase2_lr=PHASE2_LR,
    batch_size=BATCH_SIZE,
    use_focal_loss=True,
    save_name=None,
):
    """Full two-phase training pipeline.

    Returns: (model, history, best_val_acc, checkpoint_path)
    """
    if save_name is None:
        save_name = model_name

    # Datasets
    train_ds = InMemoryDataset(train_images, train_labels_list, transform=get_train_transforms())
    val_ds = InMemoryDataset(val_images, val_labels_list, transform=get_val_transforms())

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    # Model
    model = create_model(model_name, pretrained=True).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel: {model_name}")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    # Loss
    class_weights = get_class_weights(train_ds).to(device)
    if use_focal_loss:
        criterion = FocalLoss(alpha=class_weights).to(device)
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights).to(device)

    checkpoint_path = f"{save_name}_best.pth"
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    epochs_no_improve = 0

    # ── Phase 1: Head-only ────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"Phase 1: Head-only training for {phase1_epochs} epochs (lr={phase1_lr})")
    print(f"{'='*60}")
    model.freeze_backbone()
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=phase1_lr, weight_decay=WEIGHT_DECAY)

    for epoch in range(1, phase1_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, is_train=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, None, device, is_train=False)
        elapsed = time.time() - t0

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"[P1] Epoch {epoch}/{phase1_epochs} | "
              f"Train: {train_loss:.4f} / {train_acc:.4f} | "
              f"Val: {val_loss:.4f} / {val_acc:.4f} | {elapsed:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({"model_state_dict": model.state_dict(), "model_name": model_name, "epoch": epoch}, checkpoint_path)
            print(f"  -> Best val acc: {best_val_acc:.4f}, saved.")

    # ── Phase 2: Full fine-tuning ─────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"Phase 2: Full fine-tuning for {phase2_epochs} epochs (lr={phase2_lr})")
    print(f"{'='*60}")
    model.unfreeze_backbone()
    optimizer = AdamW(model.parameters(), lr=phase2_lr, weight_decay=WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=phase2_epochs)

    for epoch in range(1, phase2_epochs + 1):
        t0 = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, is_train=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, None, device, is_train=False)
        scheduler.step()
        elapsed = time.time() - t0

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        global_epoch = phase1_epochs + epoch
        print(f"[P2] Epoch {epoch}/{phase2_epochs} (g:{global_epoch}) | "
              f"Train: {train_loss:.4f} / {train_acc:.4f} | "
              f"Val: {val_loss:.4f} / {val_acc:.4f} | "
              f"LR: {scheduler.get_last_lr()[0]:.2e} | {elapsed:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save({"model_state_dict": model.state_dict(), "model_name": model_name, "epoch": global_epoch}, checkpoint_path)
            print(f"  -> Best val acc: {best_val_acc:.4f}, saved.")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping after {EARLY_STOPPING_PATIENCE} epochs without improvement.")
            break

    # Load best checkpoint
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])

    print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

    # Plot curves
    plot_training_curves(history, model_name)

    return model, history, best_val_acc, checkpoint_path


print("Training functions defined.")

In [ ]:
# ============================================================================
# Cell 11: Train EfficientNet-B3 (Primary Model)
# ============================================================================
# Phase 1: 5 epochs head-only | Phase 2: 25 epochs full fine-tuning

best_model, best_history, best_val_acc, best_checkpoint_path = train_model(
    model_name="efficientnet_b3",
    train_images=train_imgs,
    train_labels_list=train_labels,
    val_images=val_imgs,
    val_labels_list=val_labels,
    phase1_epochs=PHASE1_EPOCHS,   # 5
    phase2_epochs=PHASE2_EPOCHS,   # 25
    save_name="efficientnet_b3",
)

print(f"\nBest EfficientNet-B3 val accuracy: {best_val_acc:.4f}")
print(f"Checkpoint saved at: {best_checkpoint_path}")

In [ ]:
# ============================================================================
# Cell 12: Evaluate EfficientNet-B3 on Test Set
# ============================================================================

def evaluate_model(model, test_images, test_labels_list, model_name="Model"):
    """Evaluate model on test set with confusion matrix and classification report."""
    test_ds = InMemoryDataset(test_images, test_labels_list, transform=get_val_transforms())
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model.eval()
    all_targets, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, targets in tqdm(test_loader, desc="Evaluating"):
            images = images.to(device)
            logits = model(images)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(dim=1).cpu().tolist()
            all_targets.extend(targets.tolist())
            all_preds.extend(preds)
            all_probs.append(probs)

    all_probs_np = np.concatenate(all_probs, axis=0)

    # Metrics
    accuracy = accuracy_score(all_targets, all_preds)
    f1_macro = f1_score(all_targets, all_preds, average="macro")
    f1_weighted = f1_score(all_targets, all_preds, average="weighted")
    precision_macro = precision_score(all_targets, all_preds, average="macro")
    recall_macro = recall_score(all_targets, all_preds, average="macro")

    print(f"\n{'='*50}")
    print(f"Test Results - {model_name}")
    print(f"{'='*50}")
    print(f"Accuracy:          {accuracy:.4f}")
    print(f"F1 (macro):        {f1_macro:.4f}")
    print(f"F1 (weighted):     {f1_weighted:.4f}")
    print(f"Precision (macro): {precision_macro:.4f}")
    print(f"Recall (macro):    {recall_macro:.4f}")

    # Classification report
    report = classification_report(all_targets, all_preds, target_names=CLASS_NAMES)
    print(f"\nClassification Report:\n{report}")

    # Confusion matrix
    cm = confusion_matrix(all_targets, all_preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap="Blues")
    ax.set_title(f"{model_name} - Confusion Matrix (Test Set)")
    plt.tight_layout()
    plt.show()

    # Inference time
    dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    with torch.no_grad():
        for _ in range(10):
            model(dummy_input)  # warm-up
    times = []
    with torch.no_grad():
        for _ in range(100):
            t0 = time.perf_counter()
            model(dummy_input)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    avg_ms = np.mean(times)
    print(f"\nInference time: {avg_ms:.2f} ms/frame (avg over 100 runs)")

    total_params = sum(p.numel() for p in model.parameters())

    return {
        "model": model_name,
        "params": total_params,
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "inference_ms": avg_ms,
    }


# Evaluate EfficientNet-B3
b3_metrics = evaluate_model(best_model, test_imgs, test_labels, model_name="EfficientNet-B3")
all_results = [b3_metrics]

In [ ]:
# ============================================================================
# Cell 13: Train & Evaluate Other Models (Comparison)
# ============================================================================
# Shorter training: 3 epochs head-only + 12 epochs fine-tuning

comparison_models = ["efficientnet_b0", "mobilenet_v3", "resnet50_cbam"]

for model_name in comparison_models:
    print(f"\n{'#'*70}")
    print(f"# Training: {model_name}")
    print(f"{'#'*70}")

    model, history, val_acc, ckpt_path = train_model(
        model_name=model_name,
        train_images=train_imgs,
        train_labels_list=train_labels,
        val_images=val_imgs,
        val_labels_list=val_labels,
        phase1_epochs=3,   # shorter for comparison
        phase2_epochs=12,  # shorter for comparison
        save_name=model_name,
    )

    # Evaluate on test set
    display_name = {
        "efficientnet_b0": "EfficientNet-B0",
        "mobilenet_v3": "MobileNetV3-Large",
        "resnet50_cbam": "ResNet50+CBAM",
    }[model_name]

    metrics = evaluate_model(model, test_imgs, test_labels, model_name=display_name)
    all_results.append(metrics)

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("All models trained and evaluated!")
print(f"{'='*60}")

In [ ]:
# ============================================================================
# Cell 14: Model Comparison Table
# ============================================================================

results_df = pd.DataFrame(all_results)
results_df["params_M"] = results_df["params"] / 1e6
results_df = results_df[["model", "params_M", "accuracy", "f1_macro", "f1_weighted", "inference_ms"]]
results_df.columns = ["Model", "Params (M)", "Accuracy", "F1 (Macro)", "F1 (Weighted)", "Inference (ms)"]
results_df = results_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)

print("Model Comparison Results")
print("=" * 80)
display(results_df.style.format({
    "Params (M)": "{:.1f}",
    "Accuracy": "{:.4f}",
    "F1 (Macro)": "{:.4f}",
    "F1 (Weighted)": "{:.4f}",
    "Inference (ms)": "{:.2f}",
}).highlight_max(subset=["Accuracy", "F1 (Macro)", "F1 (Weighted)"], color="lightgreen")
 .highlight_min(subset=["Inference (ms)", "Params (M)"], color="lightyellow"))

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].barh(results_df["Model"], results_df["Accuracy"], color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
axes[0].set_xlabel("Accuracy"); axes[0].set_title("Test Accuracy")
axes[0].set_xlim(0, 1)

axes[1].barh(results_df["Model"], results_df["F1 (Macro)"], color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
axes[1].set_xlabel("F1 Score"); axes[1].set_title("F1 (Macro)")
axes[1].set_xlim(0, 1)

axes[2].barh(results_df["Model"], results_df["Inference (ms)"], color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'])
axes[2].set_xlabel("ms/frame"); axes[2].set_title("Inference Time")

plt.suptitle("Model Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Identify best model
best_idx = results_df["Accuracy"].idxmax()
print(f"\nBest model: {results_df.loc[best_idx, 'Model']} (Accuracy: {results_df.loc[best_idx, 'Accuracy']:.4f})")

In [ ]:
# ============================================================================
# Cell 15: ONNX Export
# ============================================================================

import onnx
import onnxruntime as ort

# Load best EfficientNet-B3 model for export
export_model = create_model("efficientnet_b3", pretrained=False).to("cpu")
ckpt = torch.load("efficientnet_b3_best.pth", map_location="cpu", weights_only=True)
export_model.load_state_dict(ckpt["model_state_dict"])
export_model.eval()

# Export to ONNX
onnx_path = "emotion_model.onnx"
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)

torch.onnx.export(
    export_model,
    dummy_input,
    onnx_path,
    opset_version=17,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
)

# File size
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"ONNX model exported: {onnx_path}")
print(f"Model size: {size_mb:.2f} MB")

# Validate ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model validation: PASSED")

# Compare outputs
session = ort.InferenceSession(onnx_path)
input_name = session.get_inputs()[0].name
dummy_np = dummy_input.numpy()

onnx_output = session.run(None, {input_name: dummy_np})[0]
with torch.no_grad():
    pytorch_output = export_model(dummy_input).numpy()

if np.allclose(pytorch_output, onnx_output, atol=1e-5):
    print("Output validation: PASSED (ONNX matches PyTorch, atol=1e-5)")
else:
    max_diff = np.max(np.abs(pytorch_output - onnx_output))
    print(f"Output validation: WARNING - max difference = {max_diff:.6e}")

# ONNX Runtime inference time
for _ in range(10):
    session.run(None, {input_name: dummy_np})  # warm-up

onnx_times = []
for _ in range(100):
    t0 = time.perf_counter()
    session.run(None, {input_name: dummy_np})
    onnx_times.append((time.perf_counter() - t0) * 1000)

print(f"\nONNX Runtime inference: {np.mean(onnx_times):.2f} +/- {np.std(onnx_times):.2f} ms/frame")
print(f"PyTorch GPU inference:  {b3_metrics['inference_ms']:.2f} ms/frame")

In [ ]:
# ============================================================================
# Cell 16: Save Models to Google Drive
# ============================================================================

import shutil

save_path = '/content/drive/MyDrive/Tubitak-2209B/models/'
os.makedirs(save_path, exist_ok=True)

# Save best PyTorch checkpoint
shutil.copy('efficientnet_b3_best.pth', save_path)
print(f"PyTorch checkpoint saved: {save_path}efficientnet_b3_best.pth")

# Save ONNX model
shutil.copy('emotion_model.onnx', save_path)
print(f"ONNX model saved: {save_path}emotion_model.onnx")

# Save comparison results
results_save_path = '/content/drive/MyDrive/Tubitak-2209B/results/'
os.makedirs(results_save_path, exist_ok=True)
results_df.to_csv(f'{results_save_path}model_comparison.csv', index=False)
print(f"Results saved: {results_save_path}model_comparison.csv")

# Save all other model checkpoints if they exist
for model_name in ["efficientnet_b0", "mobilenet_v3", "resnet50_cbam"]:
    ckpt_file = f"{model_name}_best.pth"
    if os.path.exists(ckpt_file):
        shutil.copy(ckpt_file, save_path)
        print(f"Saved: {save_path}{ckpt_file}")

# Provide download link (Colab only)
print("\n--- Download models locally ---")
try:
    from google.colab import files
    print("Use the files below to download:")
    # files.download('efficientnet_b3_best.pth')
    # files.download('emotion_model.onnx')
    print("Uncomment the lines above to trigger download, or access from Google Drive.")
except ImportError:
    print("Not running in Colab; models are saved locally.")

print(f"\nAll models and results saved to Google Drive: {save_path}")

# Ozet ve Sonraki Adimlar

## Tamamlanan Isler
- FER2013 veri seti 7 siniftan 3 sinifa (positive, neutral, negative) donusturuldu
- Surprise sinifi haric tutuldu
- Stratified train/val/test split yapildi (%70/%15/%15)
- 4 farkli model mimarisi egitildi ve karsilastirildi:
  - **EfficientNet-B3** (ana model, 30 epoch)
  - **EfficientNet-B0** (15 epoch)
  - **MobileNetV3-Large** (15 epoch)
  - **ResNet50+CBAM** (15 epoch)
- Focal Loss ile sinif dengesizligi ele alindi
- 2 fazli egitim stratejisi uygulandi (head-only + full fine-tuning)
- En iyi model ONNX formatina aktarildi

## Sonraki Adimlar
1. **Deploy**: ONNX modelini Flask API ile entegre et
2. **Real-time Inference**: Webcam uzerinden canli duygu analizi
3. **Dikkat Skorlama**: Duygu tahminlerini dikkat skoruna donustur
4. **Yuz Tanima**: Ogrenci bazli dikkat takibi icin yuz tanima ekle
5. **Dashboard**: Sonuclari gorsellestirmek icin web arayuzu

## Dosya Konumlari (Google Drive)
- Model: `/content/drive/MyDrive/Tubitak-2209B/models/efficientnet_b3_best.pth`
- ONNX: `/content/drive/MyDrive/Tubitak-2209B/models/emotion_model.onnx`
- Sonuclar: `/content/drive/MyDrive/Tubitak-2209B/results/model_comparison.csv`